In [2]:
# this cell contains the complete data collection preprocessing and feature transformation and selection
# pls check the attached tester file for the reasons (deep) on each step 
# also my apologies as the tester file is bit messy will push the updated one soon 
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

import matplotlib.pyplot as plt
# Loading data
return_data = pd.read_csv('ecommerce_returns_synthetic_data.csv')
fraud_data = pd.read_csv('Fraudulent_E-Commerce_Transaction_Data_2.csv')

# using fraud dataset 
data = fraud_data.copy()

#  Sorting for time-based operations 
data["Transaction Date"] = pd.to_datetime(data["Transaction Date"])
data = data.sort_values(["Customer ID", "Transaction Date"]).reset_index(drop=True)

#Handling ip addresses
location_fraud_rate = data.groupby("Customer Location")["Is Fraudulent"].mean()
data["Loc_Fraud_Rate"] = data["Customer Location"].map(location_fraud_rate)

# customer IP diversity
data["Customer_IP_Count"] = data.groupby("Customer ID")["IP Address"].transform("nunique")

# IP usage and IP-level fraud history
data["IP_Usage_Count"] = data.groupby("IP Address")["Transaction Amount"].transform("count")
data["IP_Fraud_History"] = data.groupby("IP Address")["Is Fraudulent"].transform("mean")

# drop unnnesscery transaction id 
data = data.drop(columns=["Transaction ID"])

# Working with date and time 
data["Year"] = data["Transaction Date"].dt.year
data["Month"] = data["Transaction Date"].dt.month
data["Day"] = data["Transaction Date"].dt.day
data["Weekday"] = data["Transaction Date"].dt.weekday
data["IsWeekend"] = (data["Weekday"] >= 5).astype(int)
data["Transaction Hour"] = data["Transaction Date"].dt.hour

# droping raw Transaction Date and unneeded date pieces
data = data.drop(columns=["Transaction Date", "Day", "Year"])#day and year found unuseful

#  Customer historical fraud features
# compute past fraud count (exclude current row)
data["Cust_Past_Fraud"] = data.groupby("Customer ID")["Is Fraudulent"].cumsum() - data["Is Fraudulent"]
# binary flag if any past fraud
data["Cust_Fraud_Flag"] = (data["Cust_Past_Fraud"] > 0).astype(int)
# cumulative transaction count per customer 
data["Cust_Txn_Count"] = data.groupby("Customer ID").cumcount()
# historical fraud rate 
data["Cust_Fraud_Rate"] = data["Cust_Past_Fraud"] / data["Cust_Txn_Count"].replace(0, 1)

# dropping  Customer ID (we do not use raw IDs in model)
data = data.drop(columns=["Customer ID"])

# Checking  if history features are informative; if all-zero drop them 
past_sum = data["Cust_Past_Fraud"].sum()
flag_sum = data["Cust_Fraud_Flag"].sum()
rate_sum = data["Cust_Fraud_Rate"].sum()

if (past_sum == 0) and (flag_sum == 0) and (rate_sum == 0):
    # drop uninformative history features
    data = data.drop(columns=["Cust_Past_Fraud", "Cust_Fraud_Flag", "Cust_Fraud_Rate", "Cust_Txn_Count"], errors='ignore')
else:
    # keep them
    pass

# Shipping vs Billing address 
data["Address_Mismatch"] = (data["Shipping Address"] != data["Billing Address"]).astype(int)
# remove raw address columns
data = data.drop(columns=["Shipping Address", "Billing Address"])

#one-hot encoding
# apply one-hot for Payment Method, Product Category, Device Used
data = pd.get_dummies(
    data,
    columns=[col for col in ["Payment Method", "Product Category", "Device Used"] if col in data.columns],
    drop_first=True
)

#  Location and IP cleaning
# Loc_Fraud_Rate already present  dropping  raw Customer Location
if "Customer Location" in data.columns:
    data = data.drop(columns=["Customer Location"])

# dropping IP Address after deriving IP features
if "IP Address" in data.columns:
    data = data.drop(columns=["IP Address"])

#  Numeric scaling 
from sklearn.preprocessing import StandardScaler
num_cols = [c for c in ["Transaction Amount", "Quantity", "Customer Age", "Account Age Days", "Transaction Hour"] if c in data.columns]
scaler = StandardScaler()
data[num_cols] = scaler.fit_transform(data[num_cols])

#  Convert boolean-like columns to int
bool_cols = data.select_dtypes(include='bool').columns.tolist()
data[bool_cols] = data[bool_cols].astype(int)

#  Reducing dataset for local training (keep all fraud rows + sample non-fraud) 
fraud = data[data["Is Fraudulent"] == 1]
nonfraud = data[data["Is Fraudulent"] == 0].sample(n=200_000, random_state=42)
data_small = pd.concat([fraud, nonfraud]).sample(frac=1, random_state=42).reset_index(drop=True)

#  Final checks and saving 
print("Final dataset shape (full):", data.shape)
print("Final sampled dataset shape:", data_small.shape)
print("Target distribution (sampled):")
print(data_small["Is Fraudulent"].value_counts(normalize=False))

#  saving  processed files
data.to_csv("processed_fraud_data_full.csv", index=False)
data_small.to_csv("processed_fraud_data_sample.csv", index=False)


Final dataset shape (full): (1472952, 23)
Final sampled dataset shape: (273838, 23)
Target distribution (sampled):
Is Fraudulent
0    200000
1     73838
Name: count, dtype: int64


,Transaction Amount,Quantity,Customer Age,Is Fraudulent,Account Age Days,Transaction Hour,Loc_Fraud_Rate,Customer_IP_Count,IP_Usage_Count,IP_Fraud_History,...,Address_Mismatch,Payment Method_bank transfer,Payment Method_credit card,Payment Method_debit card,Product Category_electronics,Product Category_health & beauty,Product Category_home & garden,Product Category_toys & games,Device Used_mobile,Device Used_tablet
0,-0.382471,-0.000163,0.948495,0,-1.447130,1.664742,0.058737,1,1,0.0,...,0,0,1,0,1,0,0,0,0,0
1,-0.674341,-0.000163,0.548620,0,-0.689159,-1.222287,0.083333,1,1,0.0,...,0,0,1,0,0,0,0,1,0,1
2,-0.093080,-0.000163,1.548307,0,-0.614297,-1.510990,0.054902,1,1,0.0,...,0,0,0,1,0,1,0,0,0,1
3,-0.308505,-0.000163,1.048464,0,0.368258,0.365579,0.053691,1,1,0.0,...,0,0,0,1,0,0,0,0,0,1
4,0.439881,-0.707008,-0.750973,0,1.528609,-0.933584,0.076923,1,1,0.0,...,0,1,0,0,0,0,1,0,1,0


In [ ]:
#all the training and evalatuation with detailed metrics is done in training file 

